In [9]:
import pandas as pd
import numpy as np
from pathlib import Path

# 1) Folder setup (data in / outputs out)
DATA_DIR = Path("data")
OUT_DIR = Path("outputs")

DATA_DIR.mkdir(exist_ok=True)
OUT_DIR.mkdir(exist_ok=True)


# 2) Helper functions
def clean_zip(value):
    """
    Convert a ZIP value into a clean 5-digit ZIP string.
    Returns np.nan if the value is missing or invalid.

    Handles:
    - floats like 90012.0
    - ZIP+4 like "90012-1234"
    - short ZIPs like "123" -> "00123"
    """
    if pd.isna(value):
        return np.nan

    s = str(value).strip()

    # Remove trailing ".0" if it came in as a float (common in CSVs)
    if s.endswith(".0"):
        s = s[:-2]

    # If ZIP+4 format, keep only the first 5 digits
    if "-" in s:
        s = s.split("-", 1)[0]

    # Keep only digits (in case of weird formatting like "90012 ")
    if not s.isdigit():
        return np.nan

    # Pad to 5 digits (e.g., "123" -> "00123")
    return s.zfill(5)


def find_first_col(df, candidates):
    """
    Find the first column in df that matches one of the candidate names
    (case-insensitive). Returns the actual column name or None.
    """
    # Map lowercase column name -> actual column name
    lower_to_actual = {col.lower(): col for col in df.columns}

    for name in candidates:
        key = name.lower()
        if key in lower_to_actual:
            return lower_to_actual[key]

    return None


def coerce_date(df, date_col):
    """
    Convert a date column into real datetime values.
    Invalid dates become NaT (missing).
    """
    df = df.copy()
    df[date_col] = pd.to_datetime(df[date_col], errors="coerce")
    return df


def snapshot_asof(df, zip_col="zip", date_col="date", asof="2023-05-31"):
    """
    For each ZIP, keep the most recent row where date <= asof.
    Useful when you have daily/weekly time series data and want a single snapshot.

    Steps:
    1) Drop rows missing ZIP or date
    2) Keep only rows on/before the asof date
    3) For each ZIP, take the latest (max) date row
    """
    asof = pd.Timestamp(asof)

    # Work on a clean copy
    df = df.copy()

    # Drop rows missing ZIP/date
    df = df.dropna(subset=[zip_col, date_col])

    # Keep only rows up to the asof date
    df = df[df[date_col] <= asof]

    # Sort so the latest date per ZIP is at the bottom
    df = df.sort_values([zip_col, date_col])

    # Take the last row per ZIP (latest date)
    latest_per_zip = df.groupby(zip_col, as_index=False).tail(1)

    return latest_per_zip.reset_index(drop=True)

In [10]:
DATADESK_URL = "https://raw.githubusercontent.com/datadesk/california-coronavirus-data/master/cdph-vaccination-zipcode-totals.csv"
datadesk_raw = pd.read_csv(DATADESK_URL)

datadesk_raw.shape, datadesk_raw.head(3)

((143988, 11),
          date     id       county  fips  population  partially_vaccinated  \
 0  2021-08-17  90005  Los Angeles    37     39479.0                  3436   
 1  2021-08-17  90007  Los Angeles    37     41716.0                  3659   
 2  2021-08-17  90019  Los Angeles    37     66245.0                  5182   
 
    at_least_one_dose  fully_vaccinated  partially_vaccinated_percent  \
 0              26255             22819                          0.09   
 1              30782             27123                          0.09   
 2              42474             37292                          0.08   
 
    at_least_one_dose_percent  fully_vaccinated_percent  
 0                       0.67                      0.58  
 1                       0.74                      0.65  
 2                       0.64                      0.56  )

In [11]:
datadesk = datadesk_raw.copy()

# ZIP code (id column)
datadesk["zip"] = datadesk["id"].astype(str).str.zfill(5)

# Convert date column
datadesk["date"] = pd.to_datetime(datadesk["date"])

datadesk.head()

,date,id,county,fips,population,partially_vaccinated,at_least_one_dose,fully_vaccinated,partially_vaccinated_percent,at_least_one_dose_percent,fully_vaccinated_percent,zip
0,2021-08-17,90005,Los Angeles,37,39479.0,3436,26255,22819,0.09,0.67,0.58,90005
1,2021-08-17,90007,Los Angeles,37,41716.0,3659,30782,27123,0.09,0.74,0.65,90007
2,2021-08-17,90019,Los Angeles,37,66245.0,5182,42474,37292,0.08,0.64,0.56,90019
3,2021-08-17,90024,Los Angeles,37,50288.0,2386,25869,23483,0.05,0.51,0.47,90024
4,2021-08-17,90025,Los Angeles,37,47967.0,2774,33549,30775,0.06,0.70,0.64,90025


In [12]:
datadesk_la = datadesk[datadesk["county"] == "Los Angeles"].copy()

print("Rows:", len(datadesk_la))
print("Unique ZIPs:", datadesk_la["zip"].nunique())

Rows: 25618
Unique ZIPs: 276


In [13]:
start = pd.Timestamp("2021-08-01")
end = pd.Timestamp("2023-05-31")

datadesk_la = datadesk_la[
    (datadesk_la["date"] >= start) &
    (datadesk_la["date"] <= end)
].copy()

datadesk_la["date"].min(), datadesk_la["date"].max()

(Timestamp('2021-08-17 00:00:00'), Timestamp('2023-05-30 00:00:00'))

In [18]:
datadesk_la = datadesk_la.sort_values(["zip", "date"])

datadesk_snapshot = (
    datadesk_la
    .groupby("zip", as_index=False)
    .tail(1)
)
datadesk_snapshot = datadesk_snapshot.copy()
#you can increase or decrease the number from 20 if you wanna view more data
datadesk_snapshot.head(20)

,date,id,county,fips,population,partially_vaccinated,at_least_one_dose,fully_vaccinated,partially_vaccinated_percent,at_least_one_dose_percent,fully_vaccinated_percent,zip
142387,2023-05-30,90001,Los Angeles,37,58975.0,13713,58066,44353,0.23,0.98,0.75,90001
142388,2023-05-30,90002,Los Angeles,37,53111.0,5338,40763,35425,0.10,0.77,0.67,90002
142389,2023-05-30,90003,Los Angeles,37,72741.0,8531,57564,49033,0.12,0.79,0.67,90003
142390,2023-05-30,90004,Los Angeles,37,61586.0,5969,53241,47272,0.10,0.86,0.77,90004
142391,2023-05-30,90005,Los Angeles,37,39479.0,4436,33905,29469,0.11,0.86,0.75,90005
142392,2023-05-30,90006,Los Angeles,37,61698.0,6905,50835,43930,0.11,0.82,0.71,90006
142393,2023-05-30,90007,Los Angeles,37,41716.0,6220,41143,34923,0.15,0.99,0.84,90007
142394,2023-05-30,90008,Los Angeles,37,31739.0,2531,24395,21864,0.08,0.77,0.69,90008
142395,2023-05-30,90010,Los Angeles,37,3702.0,884,5136,4252,0.24,1.39,1.15,90010
142396,2023-05-30,90011,Los Angeles,37,109414.0,15483,85612,70129,0.14,0.78,0.64,90011


In [19]:
datadesk_snapshot["vax_rate"] = pd.to_numeric(
    datadesk_snapshot["fully_vaccinated_percent"],
    errors="coerce"
)

# Remove infinite values
datadesk_snapshot["vax_rate"].replace([np.inf, -np.inf], np.nan, inplace=True)

datadesk_snapshot["vax_rate"].describe()

count    275.000000
mean       0.759891
std        0.114077
min        0.270000
25%        0.705000
50%        0.750000
75%        0.810000
max        1.660000
Name: vax_rate, dtype: float64